In [1]:
import pandas as pd
import time

file_path = "../data1.csv/data1.csv"

selected_grids = [1, 100, 1000, 5000, 10000]

chunks = []
start = time.time()

for i, chunk in enumerate(
    pd.read_csv(
        file_path,
        usecols=["GridID", "TimeInterval", "internet"],
        dtype={
            "GridID": "int32",
            "TimeInterval": "int64",
            "internet": "float32"
        },
        chunksize=200000
    )
):
    filtered = chunk[
        chunk["GridID"].isin(selected_grids)
    ]

    if not filtered.empty:
        chunks.append(filtered.copy())

    if (i + 1) % 100 == 0:
        print(f"Processed {(i + 1) * 200000:,} rows",
              flush=True)

df_selected = pd.concat(chunks, ignore_index=True)

print("Selected rows:", len(df_selected))
print("Time taken:", round(time.time() - start, 2), "seconds")

display(df_selected.head())

Processed 20,000,000 rows
Processed 40,000,000 rows
Processed 60,000,000 rows
Processed 80,000,000 rows
Processed 100,000,000 rows
Processed 120,000,000 rows
Processed 140,000,000 rows
Processed 160,000,000 rows
Selected rows: 67283
Time taken: 120.1 seconds


,GridID,TimeInterval,internet
0,1,1383260400000,NaN
1,1,1383260400000,11.028366
2,1,1383261000000,NaN
3,1,1383261000000,0.026137
4,1,1383261000000,11.100964


In [2]:
traffic = (
    df_selected
    .groupby(["GridID", "TimeInterval"], as_index=False)
    .agg(
        internet=("internet", lambda x: x.sum(min_count=1))
    )
)

traffic["datetime"] = pd.to_datetime(
    traffic["TimeInterval"],
    unit="ms",
    utc=True
)

traffic = traffic.sort_values(
    ["GridID", "datetime"]
).reset_index(drop=True)

print("Aggregated dataset shape:", traffic.shape)

display(traffic.head(10))

Aggregated dataset shape: (21600, 4)


,GridID,TimeInterval,internet,datetime
0,1,1383260400000,11.028366,2013-10-31 23:00:00+00:00
1,1,1383261000000,11.127101,2013-10-31 23:10:00+00:00
2,1,1383261600000,10.892771,2013-10-31 23:20:00+00:00
3,1,1383262200000,8.622424,2013-10-31 23:30:00+00:00
4,1,1383262800000,8.009928,2013-10-31 23:40:00+00:00
5,1,1383263400000,8.118420,2013-10-31 23:50:00+00:00
6,1,1383264000000,8.026270,2013-11-01 00:00:00+00:00
7,1,1383264600000,8.514178,2013-11-01 00:10:00+00:00
8,1,1383265200000,6.833425,2013-11-01 00:20:00+00:00
9,1,1383265800000,6.554605,2013-11-01 00:30:00+00:00


In [3]:
print("Missing internet values:", traffic["internet"].isna().sum())

print("Missing percentage:",
      round(traffic["internet"].isna().mean() * 100, 2))

print("\nMissing values per grid:")
print(traffic.groupby("GridID")["internet"]
      .apply(lambda x: x.isna().sum()))

print("\nDuplicate grid-time records:")
print(traffic.duplicated(
    subset=["GridID", "TimeInterval"]
).sum())

print("\nTime interval continuity:")
for grid_id, group in traffic.groupby("GridID"):
    gaps = (
        group.sort_values("datetime")["datetime"]
        .diff()
        .dropna()
        .ne(pd.Timedelta(minutes=10))
        .sum()
    )
    print(f"Grid {grid_id}: {gaps} irregular intervals")

Missing internet values: 0
Missing percentage: 0.0

Missing values per grid:
GridID
1        0
100      0
1000     0
5000     0
10000    0
Name: internet, dtype: int64

Duplicate grid-time records:
0

Time interval continuity:
Grid 1: 0 irregular intervals
Grid 100: 0 irregular intervals
Grid 1000: 0 irregular intervals
Grid 5000: 0 irregular intervals
Grid 10000: 0 irregular intervals


In [4]:
traffic.to_csv("../data/traffic_5grids.csv", index=False)

print("Dataset saved successfully!")

Dataset saved successfully!
